# Smart Batching Search - Testing Notebook

This notebook uses **[bigdata-smart-batching](https://pypi.org/project/bigdata-smart-batching/)** on PyPI: semantic search with intelligent company grouping, proportional sampling, and rate-limited parallel execution.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [1]:
# Library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

# Set API base URL BEFORE importing (smart_batching_config reads it at import time)
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

# Import utilities from bigdata-smart-batching package
from bigdata_smart_batching import (
    plan_search,
    execute_search,
    deduplicate_documents,
    save_plan,
    load_plan,
    load_universe_from_csv,
    convert_to_dataframe,
)

## 2. Configuration

In [2]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
TEST_TEXT = "Decline in customer confidence in the company"
TEST_UNIVERSE_CSV = "id_name_mapping_us_top_3000.csv"
# TEST_UNIVERSE_CSV = "sample_universe.csv"  # small test universe
TEST_START_DATE = "2021-01-01"
TEST_END_DATE = "2021-06-30"
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV}")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v1_UP...9T56

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'Decline in customer confidence in the company'
   Universe: id_name_mapping_us_top_3000.csv
   Date Range: 2021-01-01 to 2021-06-30
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [3]:
# Test loading universe from CSV
try:
    companies = load_universe_from_csv(TEST_UNIVERSE_CSV)
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

2026-04-17 15:57:32,991 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
✅ Loaded 4731 companies from id_name_mapping_us_top_3000.csv
   First 5 companies: ['00067A', '001F1B', '002A99', '00326D', '003B70']


## 4. Step 1: Plan Search

In [4]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)
    
    try:
        plan = plan_search(
            text=TEST_TEXT,
            universe=TEST_UNIVERSE_CSV,
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
            volume_query_mode="iterative",
            max_iterations_per_batch=10
        )
        
        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['chunk_upper_bound_estimate']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")
        
        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")
        
        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")
        
    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-04-17 15:57:32,997 - INFO - Planning search for text: 'Decline in customer confidence in the company'
2026-04-17 15:57:32,998 - INFO - Date range: 2021-01-01 to 2021-06-30
2026-04-17 15:57:33,000 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
2026-04-17 15:57:33,000 - INFO - Loaded 4731 companies from universe
2026-04-17 15:57:33,002 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting
2026-04-17 15:57:33,004 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
PHASE 1: Querying full period for all companies (2021-01-01 to 2021-06-30)
         Mode: iterative
    [ITERATIVE MODE] Querying 4731 companies in 10 batches of 500 across 1 date sub-range(s) (10 work items, max_workers=8)
    Each batch iterates until no new companies are found (max 10 iterations)
      Batch 8/10, Iter 1: Found 335 new companies, 165 remaining
    

## 5. Save Plan (Optional)

In [5]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-04-17 15:57:45,984 - INFO - Plan saved to test_search_plan.json
✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [6]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results_raw = execute_search(
            search_plan=plan,
            chunk_percentage=0.1,
            requests_per_minute=100,  # Rate limit
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
        )
        
        results = deduplicate_documents(results_raw)

        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} deduplicated chunks")
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-04-17 15:57:45,989 - INFO - Executing search with 10.0% of chunks
2026-04-17 15:57:45,990 - INFO - Total maximum expected chunks: 6,942
2026-04-17 15:57:45,990 - INFO - Searching 77 baskets
2026-04-17 15:57:47,019 - INFO - Basket basket_10_medium_20210101_20210630: Retrieved 59 documents with 70 chunks
2026-04-17 15:57:47,161 - INFO - Basket basket_15_medium_20210101_20210630: Retrieved 85 documents with 91 chunks
2026-04-17 15:57:47,229 - INFO - Basket basket_0_high_20210101_20210630: Retrieved 73 documents with 94 chunks
2026-04-17 15:57:47,291 - INFO - Basket basket_16_medium_20210101_20210630: Retrieved 69 documents with 85 chunks
2026-04-17 15:57:47,456 - INFO - Basket basket_14_medium_20210101_20210630: Retrieved 73 documents with 81 chunks
2026-04-17 15:57:47,482 - INFO - Basket basket_11_medium_20210101_20210630: Retrieved 73 documents with 88 chunks
2026-04-17 15:57:47,4

## 7. Analyze Results

In [7]:
# Convert to DataFrame (exploded by chunk)
df = convert_to_dataframe(results)
df.head()

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,url,reporting_entities
0,2021-06-26,EC9424BA15A7219134038982D33D6CE1,SHAREHOLDER ALERT: Pomerantz Law Firm Reminds ...,5A5702,Benzinga,RANK_1,7,"However, the Company failed to inform investor...",0.157730,-0.73,"[46E936, 738585, DC1A9F, C4F920, 0ABAD2, 0ABAD...",,[]
1,2021-05-13,0557E1AD83512815D5EC4D5497AFBBC8,"SHAREHOLDER ALERT: Levi & Korsinsky, LLP Notif...",5A5702,Benzinga,RANK_1,3,"However, the Company failed to inform investor...",0.122657,-0.72,"[738585, 5E647F, 5E647F, 5E647F, 0ABAD2, 0ABAD...",,[]
2,2021-06-30,AC28871347B159B6D07C457E9C3359BB,SHAREHOLDER ALERT: The Gross Law Firm Notifies...,5A5702,Benzinga,RANK_1,4,"However, the Company failed to inform investor...",0.109457,-0.73,"[0ABAD2, 0ABAD2, C4F920, 5E647F, 5E647F, 5E647...",,[]
3,2021-04-21,98406B3951EFAE703E81CC6E2E605F50,Skillz volatile after another bearish report e...,B5235B,The Fly,RANK_1,1,"Skillz, an online mobile multiplayer competiti...",0.099906,-0.79,"[253C3A, 2491BC, 1FBB18, 1FBB18, 1FBB18, 1A884...",,[]
4,2021-06-11,C21AA13BEE8CC529CD9A18286AA1A9B8,"SHAREHOLDER ALERT: Levi & Korsinsky, LLP Notif...",5A5702,Benzinga,RANK_1,3,"In reality, the Company's prospects for attain...",0.096949,-0.74,"[738585, 5E647F, 5E647F, 913660, 913660, 46E93...",,[]


In [8]:
#Analyze results
if results:

    print("📈 Results Analysis")
    print("-" * 80)
    
    # Summary stats
    n_docs = df['doc_id'].nunique()
    n_chunks = len(df)
    print(f"\n   Total: {n_docs:,} documents, {n_chunks:,} chunks")
    
    # Relevance distribution
    if 'chunk_relevance' in df.columns and df['chunk_relevance'].notna().any():
        print(f"\n   Relevance Scores:")
        print(f"     Min: {df['chunk_relevance'].min():.3f}")
        print(f"     Max: {df['chunk_relevance'].max():.3f}")
        print(f"     Avg: {df['chunk_relevance'].mean():.3f}")
    
    # Sentiment distribution
    if 'chunk_sentiment' in df.columns and df['chunk_sentiment'].notna().any():
        sentiments = df['chunk_sentiment'].dropna()
        positive = (sentiments > 0).sum()
        negative = (sentiments < 0).sum()
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    if 'source_name' in df.columns:
        source_counts = df.groupby('source_name').size().sort_values(ascending=False)
        print(f"\n   Top Sources:")
        for source, count in source_counts.head(5).items():
            print(f"     {source}: {count} chunks")
    
    # Show DataFrame info
    print(f"\n   DataFrame shape: {df.shape}")
    
    # Save results
    from datetime import datetime
    results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    df.to_json(results_file, orient='records', indent=2)
    print(f"\n💾 Saved to {results_file}")

else:
    df = None
    print("⚠️  No results to analyze")

📈 Results Analysis
--------------------------------------------------------------------------------

   Total: 4,958 documents, 5,850 chunks

   Relevance Scores:
     Min: 0.012
     Max: 0.460
     Avg: 0.061

   Sentiment Distribution:
     Positive: 2382 (40.7%)
     Negative: 3423 (58.5%)
     Neutral: 42 (0.7%)

   Top Sources:
     Factset Transcripts: 1371 chunks
     Quartr Reports: 771 chunks
     Benzinga: 750 chunks
     MT Newswires: 594 chunks
     Quartr Transcripts: 424 chunks

   DataFrame shape: (5850, 13)

💾 Saved to search_results_20260417_155833.json


## 8. Summary

In [9]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['chunk_upper_bound_estimate']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['chunk_upper_bound_estimate']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 69,426
   Baskets created: 77
✅ Execution: SUCCESS
   Chunks retrieved: 4,958
   Percentage used: 10%
   Actual vs Expected: 7.1%

Test complete!


## 9. Load Saved Plan (Optional)

In [10]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('chunk_upper_bound_estimate', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-04-17 15:58:33,058 - INFO - Plan loaded from test_search_plan.json
✅ Plan loaded from test_search_plan.json
   Total expected chunks: 69,426
   Number of baskets: 77

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)
